# Этап 6. Вклад групп признаков
Выполните ячейки сверху вниз из корня проекта в окружении этапа 5. Нужны pandas, numpy, matplotlib, lightgbm и scikit-learn. Набор кандидатов, порядок строк, параметры и число деревьев зафиксированы. Сначала воспроизводим полную модель, затем независимо переобучаем каждую сокращённую версию. Результаты сохраняются в results_stage6.

Для воспроизведения используйте LightGBM 4.6.0. Установка в среде ядра: `%pip install lightgbm==4.6.0`. После установки перезапустите ядро.

In [ ]:
from pathlib import Path
import json
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display

source = Path("results_stage4")
out = Path("results_stage6")
out.mkdir(exist_ok=True)
groups = json.loads((source / "feature_groups.json").read_text(encoding="utf-8"))
features = [field for fields in groups.values() for field in fields]
summary = pd.read_csv(source / "feature_summary.csv").set_index("период")
metrics = ["Precision@10", "Recall@10", "NDCG@10"]
k = 10

def read_split(stem, name):
    frame = pd.read_csv(source / f"{stem}.csv.gz").sort_values(["user", "item"]).reset_index(drop=True)
    sizes = frame.groupby("user").label.agg(["size", "sum"])
    observed = [len(frame), len(sizes), int(frame.label.sum())]
    reference = summary.loc[name, ["кандидатные пары", "группы", "положительные пары"]].tolist()
    if observed != reference or frame.duplicated(["user", "item"]).any():
        raise ValueError(f"{name}: размеры или ключи расходятся с данными этапа 4")
    if not (frame.label.isin([0, 1]).all() and sizes["size"].ge(k).all()
            and sizes["sum"].between(1, sizes["size"] - 1).all()
            and np.isfinite(frame[features].to_numpy()).all()):
        raise ValueError(f"{name}: проверьте признаки, метки и размеры групп")
    return frame

train = read_split("train", "Обучение")
valid = read_split("validation", "Валидация")
test = read_split("test", "Тест")
previous = Path("results_stage5")
run = json.loads((previous / "run.json").read_text(encoding="utf-8"))
if features != run["features"] or set(groups) != {"user", "item", "pair"}:
    raise ValueError("Состав признаков расходится с этапом 5")
reference = pd.read_csv(previous / "per_user.csv").query("model == 'LightGBM Ranker'").set_index("user")
print("Деревья:", run["best_trees"], "Версия LightGBM:", lgb.__version__, "На этапе 5:", run["lightgbm"])

## Расчёт метрик
Используется функция этапа 5. Контрольный пример проверяет нормировку и порядок при равных баллах.

In [ ]:
discount = 1 / np.log2(np.arange(2, k + 2))
ideal = np.r_[0.0, np.cumsum(discount)]

def evaluate(frame, scores):
    if len(scores) != len(frame) or not np.isfinite(scores).all():
        raise ValueError("Проверьте предсказания")
    ordered = frame[["user", "item", "label"]].assign(score=np.asarray(scores)).sort_values(
        ["user", "score", "item"], ascending=[True, False, True])
    top = ordered.groupby("user", sort=False).head(k).copy()
    top["rank"] = top.groupby("user").cumcount() + 1
    top["gain"] = top.label * discount[top["rank"].to_numpy() - 1]
    by_user = top.groupby("user").agg(hits=("label", "sum"), dcg=("gain", "sum"))
    by_user["relevant"] = frame.groupby("user").label.sum()
    by_user["Precision@10"] = by_user.hits / k
    by_user["Recall@10"] = by_user.hits / by_user.relevant
    by_user["NDCG@10"] = by_user.dcg / ideal[np.minimum(k, by_user.relevant.to_numpy())]
    return by_user.reset_index(), top.drop(columns="gain")

# Контроль на условном списке: один релевантный фильм на второй позиции.
toy = pd.DataFrame({"user": 1, "item": range(12), "label": [0, 1] + [0]*10})
check, _ = evaluate(toy, np.zeros(len(toy)))
np.testing.assert_allclose(check[metrics].iloc[0], [0.1, 1, 1 / np.log2(3)])

## Четыре варианта обучения
Для объединения периодов группы пользователя формируются отдельно на каждую дату. Нулевую строку таблицы задаёт повторное обучение полной модели. При расхождении её метрик по пользователям с этапом 5 выполнение остановится.

In [ ]:
merged = pd.concat([train, valid], ignore_index=True)
sizes = np.concatenate([part.groupby("user", sort=False).size().to_numpy() for part in (train, valid)])
variants = {"Полная модель": None, "Без признаков пользователя": "user",
            "Без признаков фильма": "item", "Без признаков пары": "pair"}
records = []
details = []
for name, omitted in variants.items():
    excluded = set(groups[omitted]) if omitted else set()
    columns = [col for col in features if col not in excluded]
    model = lgb.LGBMRanker(n_estimators=run["best_trees"], **run["params"])
    model.fit(merged[columns], merged.label.to_numpy(dtype=np.float32, copy=True), group=sizes)
    users, _ = evaluate(test, model.predict(test[columns]))
    users = users.set_index("user").sort_index()
    if omitted is None:
        pd.testing.assert_index_equal(users.index, reference.sort_index().index)
        np.testing.assert_allclose(users[metrics], reference.sort_index()[metrics], rtol=0, atol=1e-12,
                                   err_msg="Полная модель расходится с этапом 5")
        baseline = users["NDCG@10"].copy()
    delta = users["NDCG@10"] - baseline
    records.append({"вариант": name, "признаки": len(columns), "пользователи": len(users),
                    **users[metrics].mean().to_dict(), "изменение NDCG@10": delta.mean(),
                    "выше": int((delta > 1e-12).sum()), "ниже": int((delta < -1e-12).sum()),
                    "равно": int((delta.abs() <= 1e-12).sum())})
    details.append(users.assign(variant=name, delta_ndcg=delta).reset_index())
table = pd.DataFrame(records)
table.to_csv(out / "ablation.csv", index=False, encoding="utf-8-sig")
pd.concat(details).to_csv(out / "ablation_per_user.csv", index=False, encoding="utf-8-sig")
display(table)

## Влияние групп признаков
Изменение в абсолютных единицах NDCG@10 относительно полной модели. Отрицательное значение означает снижение при исключении группы.

In [ ]:
plot_data = table.iloc[1:]
fig, ax = plt.subplots(figsize=(8, 3.4))
delta = plot_data["изменение NDCG@10"]
ax.barh(plot_data["вариант"], delta, color="#536c80")
ax.axvline(0, color="black", linewidth=0.9)
limit = max(delta.abs().max() * 1.15, 0.001)
ax.set_xlim(-limit, limit)
ax.set_xlabel("Изменение среднего NDCG@10 относительно полной модели")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(out / "fig_2_3.png", dpi=300, bbox_inches="tight")
plt.show()

## Пример рекомендаций
Берём пользователя с наименьшим идентификатором в тесте. Для трёх исходных моделей сопоставляем выдачу с жанрами его положительно оценённых фильмов до тестовой границы. Фильм с несколькими жанрами входит в счётчик каждого своего жанра.

In [ ]:
user_id = test.user.min()
top = pd.read_csv(previous / "top10.csv")
example = top.loc[top.user.eq(user_id)].copy()
with zipfile.ZipFile("data/ml-100k.zip") as bundle:
    def read_member(filename, sep):
        with bundle.open("ml-100k/" + filename) as stream:
            return pd.read_csv(stream, sep=sep, header=None, encoding="latin-1", keep_default_na=False)
    films = read_member("u.item", "|").set_index(0)
    genres = read_member("u.genre", "|").sort_values(1)[0].tolist()
    history = read_member("u.data", "\t")
flags = films.iloc[:, 4:].copy()
flags.columns = genres
example["title"] = example.item.map(films[1])
example["genres"] = [", ".join(flags.columns[flags.loc[i].astype(bool)]) for i in example.item]
example.to_csv(out / "recommendation_example.csv", index=False, encoding="utf-8-sig")
cut = pd.Timestamp(summary.loc["Тест", "граница UTC"])
liked = history.loc[(history[0] == user_id) & (history[2] >= 4)
                   & (pd.to_datetime(history[3], unit="s", utc=True) < cut), 1]
profile = flags.loc[liked].sum().sort_values(ascending=False).rename("liked_with_genre").rename_axis("genre").reset_index()
profile["liked_total"] = len(liked)
profile["user"] = user_id
profile.to_csv(out / "example_profile.csv", index=False, encoding="utf-8-sig")
display(example, profile.head())
print("Результаты:", out.resolve())